In [52]:
# face_shape_train.py

import os
import numpy as np 
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing import image
print("All modules imported.")

All modules imported.


In [53]:
# Path to your dataset
data_dir = "../../data sheets/training_set/"

In [54]:
# Step 1: Load dataset with image generator
# datagen = ImageDataGenerator(preprocessing_function=preprocess_input)
datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    # --- ADD THESE AUGMENTATION PARAMETERS ---
    rotation_range=20,       # Randomly rotate images by up to 20 degrees
    width_shift_range=0.1,   # Randomly shift images horizontally
    height_shift_range=0.1,  # Randomly shift images vertically
    shear_range=0.1,         # Apply shear transformation
    zoom_range=0.1,          # Randomly zoom in on images
    horizontal_flip=True,    # Randomly flip images horizontally (important for faces)
    fill_mode='nearest'      # Strategy for filling in new pixels
)

batch_size = 32
img_size = (224, 224)

generator = datagen.flow_from_directory(
    data_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode="sparse",  # labels as integers
    shuffle=True
)



Found 5000 images belonging to 5 classes.


In [ ]:
# from PIL import ImageFile
# ImageFile.LOAD_TRUNCATED_IMAGES = True   # allow loading corrupted images

# base_model = VGG16(weights="imagenet", include_top=False, pooling="avg")

# features = []
# labels = []

# for i in range(len(generator)):
#     try:
#         x_batch, y_batch = generator[i]  # get batch
#         feat_batch = base_model.predict(x_batch, verbose=0)   # extract CNN features
#         features.append(feat_batch)
#         labels.append(y_batch)

#         # stop when all images processed
#         if (i + 1) * batch_size >= generator.n:
#             break

#     except Exception as e:
#         print(f"⚠️ Skipping batch {i} due to error: {e}")
#         continue

# X = np.vstack(features)
# y = np.hstack(labels)

# print("✅ Feature shape:", X.shape)
# print("✅ Labels shape:", y.shape)


In [55]:
# Step 3: Train Optimized Decision Tree using Grid Search

from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV # Import for hyperparameter tuning

# Split data into training and testing sets (assuming X and y are available from Cell 4)
# This line is kept from the original code
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the parameter grid to search
param_grid = {
    # Test various tree depths
    'max_depth': [10, 15, 20, 25], 
    # Test different settings for node splitting and leaf size
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 3, 5],
    # Test both impurity criteria
    'criterion': ['gini', 'entropy'],
    # Test with and without automatic class weight balancing (critical for imbalance)
    'class_weight': [None, 'balanced'] 
}

# Initialize the Decision Tree
dtc = DecisionTreeClassifier(random_state=42)

# Initialize GridSearchCV (using 5-fold cross-validation)
grid_search = GridSearchCV(
    estimator=dtc, 
    param_grid=param_grid, 
    cv=5,                 # 5-fold cross-validation
    scoring='accuracy', 
    verbose=1,            # Print progress during search
    n_jobs=-1             # Use all available CPU cores for speed
)

# Perform the search and fit to the training data
print("Starting Grid Search for Optimal Decision Tree Parameters...")
grid_search.fit(X_train, y_train)

# Set the global classifier variable 'clf' to the best estimator found
clf = grid_search.best_estimator_

print("\n✅ Grid Search Complete.")
print("Best Parameters Found:", grid_search.best_params_)
print("Best Cross-Validation Score:", grid_search.best_score_)
print("\nFinal Decision Tree Classifier (clf):")
print(clf)

Starting Grid Search for Optimal Decision Tree Parameters...
Fitting 5 folds for each of 144 candidates, totalling 720 fits

✅ Grid Search Complete.
Best Parameters Found: {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 10, 'min_samples_leaf': 5, 'min_samples_split': 2}
Best Cross-Validation Score: 0.28200000000000003

Final Decision Tree Classifier (clf):
DecisionTreeClassifier(class_weight='balanced', max_depth=10,
                       min_samples_leaf=5, random_state=42)


In [56]:
# Step 3: Train Decision Tree
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

clf = DecisionTreeClassifier(max_depth=20, criterion="entropy", random_state=42)
clf.fit(X_train, y_train)

,criterion,'entropy'
,splitter,'best'
,max_depth,20
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [57]:
# Step 4: Evaluate model
y_pred = clf.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.248


In [ ]:
# Step 5: Prediction on a new image
def predict_face_shape(img_path):
    img = image.load_img(img_path, target_size=img_size)
    x = image.img_to_array(img)
    x = np.expand_dims(x, axis=0)
    x = preprocess_input(x)
    feat = base_model.predict(x)
    pred = clf.predict(feat)
    class_labels = list(generator.class_indices.keys())  # e.g. ["oval","round","square"]
    return class_labels[int(pred[0])]

# Example usage
print("Predicted Shape:", predict_face_shape("../../data sheets/kirula.png"))
print("Predicted Shape:", predict_face_shape("../../data sheets/heshan.jpg"))
print("Predicted Shape:", predict_face_shape("../../data sheets/dewni.jpg"))
print("Predicted Shape:", predict_face_shape("../../data sheets/1.jpg"))


In [58]:
import joblib
import os
import tensorflow as tf
from tensorflow.keras.applications import VGG16 
from tensorflow.keras.models import save_model

# --- Configuration for Saving ---
output_dir = './saved_models'
# VGG16 will be saved as a directory (SavedModel format)
VGG_MODEL_DIR = os.path.join(output_dir, 'vgg16_features_model') 
DT_MODEL_FILE = os.path.join(output_dir, 'decision_tree_clf.pkl')

# Create the output directory relative to where the notebook is run
os.makedirs(output_dir, exist_ok=True)
print(f"Directory created: {output_dir}")

# 1. Save the Decision Tree Classifier (clf)
try:
    # Ensure 'clf' is your final trained Decision Tree object
    joblib.dump(clf, DT_MODEL_FILE)
    print(f"✅ Saved Decision Tree Classifier to {DT_MODEL_FILE}")
except NameError:
    print("❌ ERROR: 'clf' variable not found. You must run the Decision Tree training cell!")

# 2. Save the VGG16 base model (Feature Extractor)
try:
    # Assuming 'base_model' is defined in your notebook
    if 'base_model' not in locals():
        # Load VGG16 if it wasn't explicitly defined before this cell
        base_model = VGG16(weights="imagenet", include_top=False, pooling="avg") 
    
    # Use the reliable TensorFlow SavedModel format (saves to a directory)
    tf.saved_model.save(base_model, VGG_MODEL_DIR)
    print(f"✅ Saved VGG16 Base Model to directory: {VGG_MODEL_DIR}")
except NameError:
    print("❌ ERROR: 'base_model' variable not found.")
except Exception as e:
    print(f"❌ ERROR saving VGG16 model: {e}")

Directory created: ./saved_models
✅ Saved Decision Tree Classifier to ./saved_models\decision_tree_clf.pkl
❌ ERROR saving VGG16 model: this __dict__ descriptor does not support '_DictWrapper' objects


In [ ]:
import joblib
import os
import tensorflow as tf
from tensorflow.keras.applications import VGG16 
from tensorflow.keras.models import load_model # Added for clarity

# --- Configuration for Saving (UPDATED TO H5 FILE) ---
output_dir = './saved_models'
# VGG16 will now be saved as a single H5 file
VGG_MODEL_FILE = os.path.join(output_dir, 'vgg16_features.h5') 
DT_MODEL_FILE = os.path.join(output_dir, 'decision_tree_clf.pkl')

# --- FORCED VGG16 INITIALIZATION ---
try:
    # This loads VGG16 as a feature extractor.
    base_model = VGG16(weights="imagenet", include_top=False, pooling="avg")
    print("INFO: VGG16 model successfully initialized for saving.")
except Exception as e:
    print(f"❌ FATAL ERROR: Could not initialize VGG16 model: {e}")

# Create the output directory relative to where the notebook is run
os.makedirs(output_dir, exist_ok=True)
print(f"Directory created: {output_dir}")

# Print the absolute paths to confirm the saving location
absolute_dt_path = os.path.abspath(DT_MODEL_FILE)
absolute_vgg_path = os.path.abspath(VGG_MODEL_FILE)
print(f"Expected DT Path: {absolute_dt_path}")
print(f"Expected VGG Path (H5 file): {absolute_vgg_path}")


# 1. Save the Decision Tree Classifier (clf)
try:
    # This assumes 'clf' is defined in a previous cell
    joblib.dump(clf, DT_MODEL_FILE)
    print(f"✅ Saved Decision Tree Classifier to {DT_MODEL_FILE}")
except NameError:
    print("❌ ERROR: 'clf' variable not found. You must run the Decision Tree training cell!")

# 2. Save the VGG16 base model (Feature Extractor) as H5
try:
    if 'base_model' in locals():
        # Use the simple Keras model.save() to write to an H5 file
        base_model.save(VGG_MODEL_FILE)
        
        # We delete the model from memory to ensure the next run is clean
        del base_model
        print(f"✅ Saved VGG16 Base Model to FILE: {VGG_MODEL_FILE}")
    else:
        print("❌ SKIPPED: VGG16 model could not be saved because it failed initialization.")
except Exception as e:
    print(f"❌ CRITICAL ERROR saving VGG16 model (H5 attempt failed): {e}")


INFO: VGG16 model successfully initialized for saving.
Directory created: ./../../saved_models
Expected DT Path: d:\education\japura\3rd year\1st sem\Machine Learning\project\HairstylePredicAccordingToFaceshape\saved_models\decision_tree_clf.pkl
Expected VGG Dir: d:\education\japura\3rd year\1st sem\Machine Learning\project\HairstylePredicAccordingToFaceshape\saved_models\vgg16_features_model
✅ Saved Decision Tree Classifier to ./../../saved_models\decision_tree_clf.pkl
❌ ERROR saving VGG16 model: The `save_format` argument is deprecated in Keras 3. Please remove this argument and pass a file path with either `.keras` or `.h5` extension.Received: save_format=tf


In [ ]:
import cv2
import numpy as np
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.vgg16 import preprocess_input

# --- Live prediction function ---
def live_prediction():
    cap = cv2.VideoCapture(0)  # 0 = default webcam

    class_labels = list(generator.class_indices.keys())  # e.g. ["oval","round","square"]

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # Preprocess frame for VGG16
        img = cv2.resize(frame, img_size)               # resize to target size
        x = image.img_to_array(img)
        x = np.expand_dims(x, axis=0)
        x = preprocess_input(x)

        # Feature extraction + prediction
        feat = base_model.predict(x, verbose=0)
        pred = clf.predict(feat)
        face_shape = class_labels[int(pred[0])]

        # Display prediction on the frame
        cv2.putText(frame, f"Shape: {face_shape}", (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

        cv2.imshow("Live Face Shape Prediction", frame)

        # Press 'q' to quit
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    cap.release()
    cv2.destroyAllWindows()

# --- Run live camera prediction ---
live_prediction()


In [ ]:
import cv2
import numpy as np
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.vgg16 import preprocess_input
import threading
import tkinter as tk

# Global flag to stop the camera loop
stop_camera = False

# Haarcascade for face detection
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

# --- Function to stop camera ---
def stop():
    global stop_camera
    stop_camera = True

# --- Live prediction function ---
def live_prediction():
    global stop_camera
    cap = cv2.VideoCapture(0)
    class_labels = list(generator.class_indices.keys())  # e.g. ["oval","round","square"]

    while not stop_camera:
        ret, frame = cap.read()
        if not ret:
            break

        # Convert to grayscale for face detection
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5)

        for (x, y, w, h) in faces:
            # Draw rectangle around face
            cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 255), 2)

            # Crop face for prediction
            face_roi = frame[y:y + h, x:x + w]
            if face_roi.size == 0:
                continue

            # Preprocess for VGG16
            img_input = cv2.resize(face_roi, img_size)
            x_input = image.img_to_array(img_input)
            x_input = np.expand_dims(x_input, axis=0)
            x_input = preprocess_input(x_input)

            # Feature extraction + prediction
            feat = base_model.predict(x_input, verbose=0)
            pred = clf.predict(feat)
            face_shape = class_labels[int(pred[0])]

            # Display prediction
            cv2.putText(frame, f"{face_shape}", (x, y - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)

        cv2.imshow("Live Face Shape Prediction", frame)

        # Allow closing by pressing 'q'
        if cv2.waitKey(1) & 0xFF == ord('q'):
            stop_camera = True
            break

    cap.release()
    cv2.destroyAllWindows()

# --- Tkinter GUI with Stop button ---
root = tk.Tk()
root.title("Face Shape Scanner")
root.geometry("200x100")

stop_button = tk.Button(root, text="Stop Camera", command=stop, bg="red", fg="white", font=("Arial", 14))
stop_button.pack(expand=True, fill="both")

# Run camera in a separate thread
camera_thread = threading.Thread(target=live_prediction)
camera_thread.start()

root.mainloop()
stop_camera = True
camera_thread.join()
